In [ ]:
#imports
import pandas as pd
import numpy as np
import re
import html
import unicodedata
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

## 1. Initialisation
This file to be in the same directory as the raw data file "IMBD_reviews.json"

In [3]:
# Read review dataset
df = pd.read_json("IMDB_reviews.json", lines=True)

## 2. Analysis on the dataset

In [4]:
# Check details of reviews dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 573913 entries, 0 to 573912
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   review_date     573913 non-null  str  
 1   movie_id        573913 non-null  str  
 2   user_id         573913 non-null  str  
 3   is_spoiler      573913 non-null  bool 
 4   review_text     573913 non-null  str  
 5   rating          573913 non-null  int64
 6   review_summary  573913 non-null  str  
dtypes: bool(1), int64(1), str(5)
memory usage: 26.8 MB


In [5]:
print(df.isnull().sum())
# No sign of missing entries initially, but requires further checks into empty strings

review_date       0
movie_id          0
user_id           0
is_spoiler        0
review_text       0
rating            0
review_summary    0
dtype: int64


In [6]:
print(df.shape)
# 573,913 reviews, 7 features

(573913, 7)


In [7]:
print(df.columns)

Index(['review_date', 'movie_id', 'user_id', 'is_spoiler', 'review_text',
       'rating', 'review_summary'],
      dtype='str')


In [8]:
print(df["review_text"].str.len().describe())
# Mean character length is 1460, relatively large inputs

count    573913.000000
mean       1460.553525
std        1125.577019
min          18.000000
25%         719.000000
50%        1052.000000
75%        1815.000000
max       14963.000000
Name: review_text, dtype: float64


In [13]:
print(df["is_spoiler"].value_counts(normalize=True))
# Dataset is slightly imbalanced (73.7% non-spoiler vs 26.3% spoiler)

is_spoiler
False    0.737026
True     0.262974
Name: proportion, dtype: float64


In [9]:
print((df["review_text"].str.strip() == "").sum())
# No empty or whitespace-only reviews detected, so no need for removal of trivial empty entries

0


In [10]:
print(df.duplicated(subset=["review_text"]).sum())
# 528 duplicate review texts, redundant samples that may cause bias or data leakage if not removed

528


In [11]:
dup_text = df[df.duplicated(subset=["review_text"], keep=False)]
dup_text.groupby("review_text").size().sort_values(ascending=False).head(10)
# Some duplicated text are quite long, indicating repeated reviews, should remove

review_text
I have never seen such an amazing film since I saw The Shawshank Redemption. Shawshank encompasses friendships, hardships, hopes, and dreams. And what is so great about the movie is that it moves you, it gives you hope. Even though the circumstances between the characters and the viewers are quite different, you don't feel that far removed from what the characters are going through.It is a simple film, yet it has an everlasting message. Frank Darabont didn't need to put any kind of outlandish special effects to get us to love this film, the narration and the acting does that for him. Why this movie didn't win all seven Oscars is beyond me, but don't let that sway you to not see this film, let its ranking on the IMDb's top 250 list sway you, let your friends recommendation about the movie sway you.Set aside a little over two hours tonight and rent this movie. You will finally understand what everyone is talking about and you will understand why this is my all time favorite m

In [12]:
print(df.groupby("review_text")["is_spoiler"].nunique().value_counts())
# Almost all duplicate reviews have consistent labels (573,307), but 78 cases show conflicting labels, minor label noise

is_spoiler
1    573307
2        78
Name: count, dtype: int64


In [13]:
df["length"] = df["review_text"].str.len()
print(df.groupby("is_spoiler")["length"].describe())
# Spoiler reviews are significantly longer on average (~1888 vs ~1308 chars), suggesting length is a strong predictive signal

               count         mean          std   min    25%     50%     75%  \
is_spoiler                                                                    
False       422989.0  1308.154425  1021.050440  18.0  682.0   947.0  1575.0   
True        150924.0  1887.676731  1283.848098  50.0  920.0  1459.0  2456.0   

                max  
is_spoiler           
False       14963.0  
True        14302.0  


In [14]:
print(df.groupby("is_spoiler")["rating"].mean())
# Spoiler reviews tend to have lower average ratings (~6.52 vs ~7.11), indicating a potential correlation between sentiment and spoiler likelihood

is_spoiler
False    7.110031
True     6.517665
Name: rating, dtype: float64


In [15]:
print(df["movie_id"].value_counts().describe())
# Review distribution is highly uneven across movies (max ~4845 reviews), indicating potential movie-level bias in the dataset

count    1572.000000
mean      365.084606
std       283.088117
min         2.000000
25%       165.000000
50%       326.000000
75%       529.000000
max      4845.000000
Name: count, dtype: float64


In [16]:
print(df["user_id"].value_counts().describe())
# Most users contribute only one review, but a few users are highly active (up to 1303 reviews), introducing potential user-level bias

count    263407.000000
mean          2.178807
std          10.665784
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        1303.000000
Name: count, dtype: float64


In [17]:
print((df["review_text"] == df["review_summary"]).sum())
# Review text and summary are almost always different (only 1 identical case), indicating they provide complementary information

1


In [23]:
print(df[df["review_text"].str.len() > 10000].shape)
# Only 25 reviews exceed 10,000 characters, indicating rare but extreme outliers that may impact model efficiency

(25, 8)


In [24]:
print(df["review_text"].str.contains("dies|ending|killer", case=False).groupby(df["is_spoiler"]).mean())
# Spoiler reviews are more likely to contain keywords like "dies", "ending", or "killer" (~27% vs ~17%), indicating partial keyword-driven signal.

is_spoiler
False    0.168489
True     0.271534
Name: review_text, dtype: float64


In [25]:
# Review date stored as object, convert to datetime
df["review_date"] = pd.to_datetime(df["review_date"])

## 3. Master model
The subsequent steps are to create the basline/master model, the canonical dataset used across all models

In [18]:
#create master dataset
df_master = df.copy()

In [19]:
# Remove duplicate reviews which have different label assigned to the same text
conflicts = df_master.groupby("review_text")["is_spoiler"].nunique()
conflicts = conflicts[conflicts > 1].index

df_master = df_master[~df_master["review_text"].isin(conflicts)]

# Remove exact duplicate reviews to avoid redundancy and data leakage
df_master = df_master.drop_duplicates(subset=["review_text"])

# Remove extremely short reviews that are likely low-information or noisy
df_master = df_master[df_master["review_text"].str.len() > 25]

# Remove extremely long reviews that may distort feature distributions and increase computational cost
df_master = df_master[df_master["review_text"].str.len() < 10000]

# Add review length as a feature due to strong correlation with spoiler labels
df_master["length"] = df_master["review_text"].str.len()

# Reset index after row removals to keep the cleaned dataset tidy and consistent
df_master = df_master.reset_index(drop=True)

In [20]:
print(df_master.shape)
# Removed 634 reviews
# Additional feature is review length "length"

(573279, 8)


In [21]:
# Re-check class distribution after cleaning to ensure preprocessing did not heavily distort label balance
print(df_master["is_spoiler"].value_counts(normalize=True))
# Original: (73.7% non-spoiler vs 26.3% spoiler), class distribution remained very similar

is_spoiler
False    0.737009
True     0.262991
Name: proportion, dtype: float64


In [30]:
# Verify that exact duplicate reviews have been fully removed
print(df_master.duplicated(subset=["review_text"]).sum())
# No duplicates left

0


In [31]:
# Verify that no identical review text remains with conflicting spoiler labels
print(df_master.groupby("review_text")["is_spoiler"].nunique().value_counts())
# All review_texts have only 1 label

is_spoiler
1    573279
Name: count, dtype: int64


In [22]:
# Basic data cleaning function
def basic_clean(text):
    text = str(text) # Forces text to be a String
    text = html.unescape(text) # Converts HTML to normal text
    text = unicodedata.normalize("NFKC", text) # Normalizes unicode into a consistent canonical form
    text = re.sub(r"<[^>]+>", " ", text) # Removes HTML tags 
    text = text.lower() # Converts all text to lowercase
    text = re.sub(r"\s+", " ", text) # Collapses all whitespace into a single space
    text = text.strip() # Removes whitespace in front and behind of text
    return text

In [23]:
# Clean "clean_text" in master dataframe
df_master["clean_text"] = df_master["review_text"].apply(basic_clean)

In [24]:
# Compare clean_text to review_text
pd.set_option("display.max_colwidth", None)
df_master[["review_text", "clean_text"]].sample(3, random_state=42)

,review_text,clean_text
450069,"I'm sorry. I wanted to like it so much.I've been a fan of the show for years. I've watched every episode 10 times. I was so incredibly excited when I heard a movie was being made. Unfortunately, it wasn't the movie I was hoping to see.It's great to hear Carrie's voice again, talking about her and her friends little dramas. But even SATC got 'carried away' with that Hollywood cheesiness. Where the show was spicy and provocative, the movie was too long and moralizing. Though some of the dialogs where pretty good, a lot of others weren't. The half of the time I felt like I was watching 4 totally different people than I saw and the show 4 years ago. Also, the story was pitiful (Carrie marries Big - oh no, she doesn't! - wait, she does!), they just tried to squeeze every possible scenario in 144 minutes (alot of breaking up and getting back together). And even my two favorite gay men had like two lines in the whole movie! The only great thing in the movie was the wardrobe. Compliment to Patricia Field, she has done it again! (Though the designer logos were shooting of the screen). But when the critics said this movie was made 'just for the fans', they were wrong. This movie was made for people who have no clue of what Sex And The City is, and just want a fun girls night out. The fans however, will be disappointed.I'm sorry. I wanted to like it so much (I rate it '10' just on my personal account, so it wouldn't be suck on a 3,5, cause it doesn't deserve that either), but Sex and the City is just a mess of very very good fashion, and bad fart jokes. But I will buy it on DVD. Just to finish my collection.","i'm sorry. i wanted to like it so much.i've been a fan of the show for years. i've watched every episode 10 times. i was so incredibly excited when i heard a movie was being made. unfortunately, it wasn't the movie i was hoping to see.it's great to hear carrie's voice again, talking about her and her friends little dramas. but even satc got 'carried away' with that hollywood cheesiness. where the show was spicy and provocative, the movie was too long and moralizing. though some of the dialogs where pretty good, a lot of others weren't. the half of the time i felt like i was watching 4 totally different people than i saw and the show 4 years ago. also, the story was pitiful (carrie marries big - oh no, she doesn't! - wait, she does!), they just tried to squeeze every possible scenario in 144 minutes (alot of breaking up and getting back together). and even my two favorite gay men had like two lines in the whole movie! the only great thing in the movie was the wardrobe. compliment to patricia field, she has done it again! (though the designer logos were shooting of the screen). but when the critics said this movie was made 'just for the fans', they were wrong. this movie was made for people who have no clue of what sex and the city is, and just want a fun girls night out. the fans however, will be disappointed.i'm sorry. i wanted to like it so much (i rate it '10' just on my personal account, so it wouldn't be suck on a 3,5, cause it doesn't deserve that either), but sex and the city is just a mess of very very good fashion, and bad fart jokes. but i will buy it on dvd. just to finish my collection."
495181,"20 years after this movie came out and it can still surprise you. I first watched about two or three years ago and I very much liked it. What really surprised me is that even the bad guy, Dr. Varnick, made me laugh. This movie is funny and appropriate for children. It does contain some things that parents might consider inappropriate for children who are very little or who are sensitive. Some material might be frightening, but overall it's a nice movie. It will make you laugh, especially the dog-nappers.The Saint Bernard became the center of attention of the whole family.At the beginning I thought that the family wouldn't accept the dog, but I kept my hopes up. I would recommend this movie to anyon

Split data by movies so that the model doesnt try to learn movie specific context to detect spoilers.  
Train/test split is group-based (by `movie_id`) and **CV is Stratified K-Fold** on the training portion.  
Approximate Split Proportion:  
Training: 0.80  
Testing: 0.20

In [25]:
# Function for splitting data into training and testing sets
def group_data_split(df, group_col="movie_id", test_size=0.2, random_state=42):
    """
    Split a dataframe into train and test sets using group-based splitting.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    group_col : str, default="movie_id"
        Column used to define groups that must not be split across train and test.
    test_size : float, default=0.2
        Proportion of groups to place in the test set.
    random_state : int, default=42
        Random seed for reproducibility.

    Returns
    -------
    df_train : pandas.DataFrame
        Training subset.
    df_test : pandas.DataFrame
        Test subset.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)

    train_idx, test_idx = next(gss.split(df, groups=df[group_col]))

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)

    return df_train, df_test

In [26]:
# Splitting data into training set and testing set (grouped by movie_id)
df_train, df_test = group_data_split(
    df_master, 
    group_col="movie_id", 
    test_size=0.2, 
    random_state=42
)

In [ ]:
# Create CV splits on the training set (group-aware by movie_id)
# This prevents the same movie_id from appearing in both fold-train and fold-val.
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
CV_SEED = 42

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
cv_splits = list(
    cv.split(
        df_train["clean_text"],
        df_train["is_spoiler"],
        groups=df_train["movie_id"],
    )
)
print("Number of CV folds:", len(cv_splits))

Number of CV folds: 5


In [ ]:
# Sanity checks for data splitting
# No leakage between train and test (movie-level)
assert set(df_train["movie_id"]).isdisjoint(df_test["movie_id"])

# Check approximate sizes
print("Train size:", len(df_train))
print("Test size:", len(df_test))
print("Train proportion:", len(df_train) / len(df_master))
print("Test proportion:", len(df_test) / len(df_master))

# Check class balance
print("\nSpoiler rate")
print("Full  :", df_master["is_spoiler"].mean())
print("Train :", df_train["is_spoiler"].mean())
print("Test  :", df_test["is_spoiler"].mean())

# Check number of unique movies
print("\nUnique movies")
print("Full  :", df_master["movie_id"].nunique())
print("Train :", df_train["movie_id"].nunique())
print("Test  :", df_test["movie_id"].nunique())

# Check fold sizes, balance, and movie-level leakage
for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
    y_tr = df_train.iloc[train_idx]["is_spoiler"]
    y_val = df_train.iloc[val_idx]["is_spoiler"]

    tr_movies = set(df_train.iloc[train_idx]["movie_id"])
    val_movies = set(df_train.iloc[val_idx]["movie_id"])
    assert tr_movies.isdisjoint(val_movies), f"Movie leakage in fold {i}"

    print(
        f"Fold {i}: train={len(train_idx)}, val={len(val_idx)}, "
        f"train_spoiler={y_tr.mean():.3f}, val_spoiler={y_val.mean():.3f}, "
        f"train_movies={len(tr_movies)}, val_movies={len(val_movies)}"
    )

NameError: name 'df_train' is not defined

## 4. Feature engineering for fixed splits
This section creates multiple fixed train/val/test splits (grouped by `movie_id`) and builds three feature sets per split: TF-IDF, TF-IDF + lemma, and BERT embeddings. Features are saved to disk for later model training and CV comparisons.

In [ ]:
# Feature engineering: TF-IDF (no lemma) for CV folds and full train/test
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib

OUTPUT_DIR = Path("artifacts/features/cv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "clean_text"
LABEL_COL = "is_spoiler"
GROUP_COL = "movie_id"


def clean_corpus(texts):
    if TEXT_COL == "clean_text":
        return [str(t) for t in texts]
    return [basic_clean(t) for t in texts]


def make_vectorizer():
    # float32 saves memory and is usually fine for TF-IDF.
    return TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        dtype=np.float32,
    )


def fit_transform_vectorizer(texts):
    vec = make_vectorizer()
    X = vec.fit_transform(clean_corpus(texts))
    return vec, X


def transform_vectorizer(texts, vec):
    return vec.transform(clean_corpus(texts))


def build_cv_fold_tfidf(df_train, fold_idx, train_idx, val_idx):
    fold_tag = f"fold{fold_idx}"
    df_tr = df_train.iloc[train_idx].reset_index(drop=True)
    df_val = df_train.iloc[val_idx].reset_index(drop=True)

    # Labels for this fold
    np.save(OUTPUT_DIR / f"labels_train_{fold_tag}.npy", df_tr[LABEL_COL].to_numpy(dtype=np.int64))
    np.save(OUTPUT_DIR / f"labels_val_{fold_tag}.npy", df_val[LABEL_COL].to_numpy(dtype=np.int64))

    # TF-IDF (fit+transform once for train; only transform for val)
    tfidf_vec, X_tr = fit_transform_vectorizer(df_tr[TEXT_COL])
    X_val = transform_vectorizer(df_val[TEXT_COL], tfidf_vec)

    joblib.dump(tfidf_vec, OUTPUT_DIR / f"tfidf_vectorizer_{fold_tag}.joblib")
    sparse.save_npz(OUTPUT_DIR / f"X_tfidf_train_{fold_tag}.npz", X_tr)
    sparse.save_npz(OUTPUT_DIR / f"X_tfidf_val_{fold_tag}.npz", X_val)


def build_full_train_test_tfidf(df_train, df_test):
    # Labels
    np.save(OUTPUT_DIR / "labels_train_full.npy", df_train[LABEL_COL].to_numpy(dtype=np.int64))
    np.save(OUTPUT_DIR / "labels_test.npy", df_test[LABEL_COL].to_numpy(dtype=np.int64))

    # TF-IDF (fit+transform once on full training set)
    tfidf_vec, X_train = fit_transform_vectorizer(df_train[TEXT_COL])
    X_test = transform_vectorizer(df_test[TEXT_COL], tfidf_vec)

    joblib.dump(tfidf_vec, OUTPUT_DIR / "tfidf_vectorizer_full.joblib")
    sparse.save_npz(OUTPUT_DIR / "X_tfidf_train_full.npz", X_train)
    sparse.save_npz(OUTPUT_DIR / "X_tfidf_test.npz", X_test)


def build_cv_tfidf_features(df_train, df_test, cv_splits):
    for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
        print(f"[tfidf] fold {i}/{len(cv_splits)}")
        build_cv_fold_tfidf(df_train, i, train_idx, val_idx)
    print("[tfidf] full train/test")
    build_full_train_test_tfidf(df_train, df_test)


build_cv_tfidf_features(df_train, df_test, cv_splits)


KeyboardInterrupt



In [ ]:
# Feature engineering: TF-IDF + lemma for CV folds and full train/test
# NOTE: The main bottleneck is lemmatization. This version lemmatizes ONCE (cached)
# and reuses the lemmatized strings across all CV folds/transforms.

from pathlib import Path
import re
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

# Reuse OUTPUT_DIR / TEXT_COL / LABEL_COL / df_train / df_test / cv_splits defined above.
# Provide a small fallback so this cell is still runnable if executed out-of-order.
try:
    clean_corpus  # noqa: B018
except NameError:
    def clean_corpus(texts):
        return [str(t) for t in texts]

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def ensure_nltk_resources():
    import nltk
    # Only required for WordNetLemmatizer.
    resources = ["wordnet", "omw-1.4"]
    for res in resources:
        try:
            nltk.data.find(f"corpora/{res}")
        except LookupError:
            nltk.download(res, quiet=True)
    return nltk

nltk = ensure_nltk_resources()
from nltk.stem import WordNetLemmatizer

_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def _lemmatize_batch(text_batch):
    lem = WordNetLemmatizer()
    out = []
    for text in text_batch:
        tokens = _TOKEN_RE.findall(str(text))
        out.append(" ".join(lem.lemmatize(tok) for tok in tokens))
    return out

def lemmatize_texts_cached(texts, cache_path: Path, n_jobs: int = -1, batch_size: int = 2000):
    """Return np.ndarray[str] of lemmatized texts, cached to disk."""
    if cache_path.exists():
        print(f"[lemma] cache hit: {cache_path}")
        return joblib.load(cache_path)

    print(f"[lemma] building cache: {cache_path} (this can take a while)")

    # Clean first (handles NaNs, non-strings)
    cleaned = clean_corpus(texts)

    # Batch to avoid joblib overhead per document (important for large corpora)
    batches = [cleaned[i : i + batch_size] for i in range(0, len(cleaned), batch_size)]

    from joblib import Parallel, delayed

    parts = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(_lemmatize_batch)(b) for b in batches
    )

    # Flatten list-of-lists
    lemmatized = np.asarray([t for part in parts for t in part], dtype=object)

    joblib.dump(lemmatized, cache_path, compress=3)
    print(f"[lemma] saved cache: {cache_path}")
    return lemmatized

# Precompute lemma text ONCE for full train/test (reused for every fold)
lemma_train = lemmatize_texts_cached(
    df_train[TEXT_COL],
    CACHE_DIR / f"lemma_train_{TEXT_COL}.joblib",
)
lemma_test = lemmatize_texts_cached(
    df_test[TEXT_COL],
    CACHE_DIR / f"lemma_test_{TEXT_COL}.joblib",
)

def make_lemma_vectorizer():
    # dtype float32 to reduce memory; keep same settings as base TF-IDF.
    return TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        dtype=np.float32,
    )

def build_cv_fold_tfidf_lemma(fold_idx, train_idx, val_idx):
    fold_tag = f"fold{fold_idx}"

    vec = make_lemma_vectorizer()

    X_tr_l = vec.fit_transform(lemma_train[train_idx])
    X_val_l = vec.transform(lemma_train[val_idx])

    joblib.dump(vec, OUTPUT_DIR / f"tfidf_lemma_vectorizer_{fold_tag}.joblib")
    sparse.save_npz(OUTPUT_DIR / f"X_tfidf_lemma_train_{fold_tag}.npz", X_tr_l)
    sparse.save_npz(OUTPUT_DIR / f"X_tfidf_lemma_val_{fold_tag}.npz", X_val_l)

def build_full_train_test_tfidf_lemma():
    vec = make_lemma_vectorizer()

    X_train_l = vec.fit_transform(lemma_train)
    X_test_l = vec.transform(lemma_test)

    joblib.dump(vec, OUTPUT_DIR / "tfidf_lemma_vectorizer_full.joblib")
    sparse.save_npz(OUTPUT_DIR / "X_tfidf_lemma_train_full.npz", X_train_l)
    sparse.save_npz(OUTPUT_DIR / "X_tfidf_lemma_test.npz", X_test_l)

def build_cv_tfidf_lemma_features(cv_splits):
    for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
        print(f"[tfidf+lemma] fold {i}/{len(cv_splits)}")
        build_cv_fold_tfidf_lemma(i, train_idx, val_idx)
    print("[tfidf+lemma] full train/test")
    build_full_train_test_tfidf_lemma()

build_cv_tfidf_lemma_features(cv_splits)

In [ ]:
# Feature engineering: BERT embeddings for CV folds and full train/test
from sentence_transformers import SentenceTransformer
import numpy as np

BERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BERT_BATCH_SIZE = 64
bert_model = SentenceTransformer(BERT_MODEL_NAME)

def encode_bert(texts):
    return bert_model.encode(
        clean_corpus(texts),
        batch_size=BERT_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

def build_cv_fold_bert(df_train, fold_idx, train_idx, val_idx):
    fold_tag = f"fold{fold_idx}"
    df_tr = df_train.iloc[train_idx].reset_index(drop=True)
    df_val = df_train.iloc[val_idx].reset_index(drop=True)

    X_tr_b = encode_bert(df_tr[TEXT_COL])
    X_val_b = encode_bert(df_val[TEXT_COL])
    np.save(OUTPUT_DIR / f"X_bert_train_{fold_tag}.npy", X_tr_b)
    np.save(OUTPUT_DIR / f"X_bert_val_{fold_tag}.npy", X_val_b)

def build_full_train_test_bert(df_train, df_test):
    X_train_b = encode_bert(df_train[TEXT_COL])
    X_test_b = encode_bert(df_test[TEXT_COL])
    np.save(OUTPUT_DIR / "X_bert_train_full.npy", X_train_b)
    np.save(OUTPUT_DIR / "X_bert_test.npy", X_test_b)

def build_cv_bert_features(df_train, df_test, cv_splits):
    for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
        build_cv_fold_bert(df_train, i, train_idx, val_idx)
    build_full_train_test_bert(df_train, df_test)

build_cv_bert_features(df_train, df_test, cv_splits)

### Saved artifacts (per CV fold)
Each fold (e.g., `fold1`) produces:
- `labels_train_fold1.npy`, `labels_val_fold1.npy`
- `X_tfidf_train_fold1.npz`, `X_tfidf_val_fold1.npz` + `tfidf_vectorizer_fold1.joblib`
- `X_tfidf_lemma_train_fold1.npz`, `X_tfidf_lemma_val_fold1.npz` + `tfidf_lemma_vectorizer_fold1.joblib`
- `X_bert_train_fold1.npy`, `X_bert_val_fold1.npy`

### Saved artifacts (full train/test)
Once CV is done, the notebook also saves full-train/test features for final model training:
- `labels_train_full.npy`, `labels_test.npy`
- `X_tfidf_train_full.npz`, `X_tfidf_test.npz` + `tfidf_vectorizer_full.joblib`
- `X_tfidf_lemma_train_full.npz`, `X_tfidf_lemma_test.npz` + `tfidf_lemma_vectorizer_full.joblib`
- `X_bert_train_full.npy`, `X_bert_test.npy`

### Lemma cache (speed-up)
- `cache/lemma_train_clean_text.joblib`, `cache/lemma_test_clean_text.joblib` (names depend on `TEXT_COL`)

All files are written to `artifacts/features/cv/`.

## 5. Load saved splits and features
Run the next cell to load train/val/test splits and feature artifacts from disk so other models can reuse them directly in this notebook.

In [ ]:
# Load CV artifacts into memory for downstream model training
from pathlib import Path
import numpy as np
import joblib
import pandas as pd
from scipy import sparse

ARTIFACT_DIR = Path("artifacts/features/cv")
N_SPLITS = 5

def load_cv_fold(fold_idx):
    fold_tag = f"fold{fold_idx}"
    artifacts = {}
    artifacts["y_train"] = np.load(ARTIFACT_DIR / f"labels_train_{fold_tag}.npy")
    artifacts["y_val"] = np.load(ARTIFACT_DIR / f"labels_val_{fold_tag}.npy")
    artifacts["X_tfidf_train"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_train_{fold_tag}.npz")
    artifacts["X_tfidf_val"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_val_{fold_tag}.npz")
    artifacts["tfidf_vectorizer"] = joblib.load(ARTIFACT_DIR / f"tfidf_vectorizer_{fold_tag}.joblib")
    artifacts["X_tfidf_lemma_train"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_train_{fold_tag}.npz")
    artifacts["X_tfidf_lemma_val"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_val_{fold_tag}.npz")
    artifacts["tfidf_lemma_vectorizer"] = joblib.load(ARTIFACT_DIR / f"tfidf_lemma_vectorizer_{fold_tag}.joblib")
    artifacts["X_bert_train"] = np.load(ARTIFACT_DIR / f"X_bert_train_{fold_tag}.npy")
    artifacts["X_bert_val"] = np.load(ARTIFACT_DIR / f"X_bert_val_{fold_tag}.npy")
    return artifacts

def load_full_train_test():
    artifacts = {}
    artifacts["y_train_full"] = np.load(ARTIFACT_DIR / "labels_train_full.npy")
    artifacts["y_test"] = np.load(ARTIFACT_DIR / "labels_test.npy")
    artifacts["X_tfidf_train_full"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_train_full.npz")
    artifacts["X_tfidf_test"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_test.npz")
    artifacts["tfidf_vectorizer_full"] = joblib.load(ARTIFACT_DIR / "tfidf_vectorizer_full.joblib")
    artifacts["X_tfidf_lemma_train_full"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_train_full.npz")
    artifacts["X_tfidf_lemma_test"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_test.npz")
    artifacts["tfidf_lemma_vectorizer_full"] = joblib.load(ARTIFACT_DIR / "tfidf_lemma_vectorizer_full.joblib")
    artifacts["X_bert_train_full"] = np.load(ARTIFACT_DIR / "X_bert_train_full.npy")
    artifacts["X_bert_test"] = np.load(ARTIFACT_DIR / "X_bert_test.npy")
    return artifacts

cv_folds = {f"fold{i}": load_cv_fold(i) for i in range(1, N_SPLITS + 1)}
full_train_test = load_full_train_test()
print("Loaded CV folds:", list(cv_folds.keys()))

For future work, ensure data is not leaked from train to val and from val to test.
df_train → training + optional cross validation
df_val   → use to compare between different models
df_test  → final evaluation (once)

If using k-fold:
Step 1: CV on df_train → choose hyperparameters
Step 2: Train final model on full df_train
Step 3: Evaluate on df_val

# Logistic Regression 

## 1. Logistic Regression on un-Lemmatised TF-IDF

In [3]:
from pathlib import Path
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

ARTIFACT_DIR = Path("artifacts/features/cv")
N_SPLITS = 5
C_GRID = [0.01, 0.1, 1.0, 10.0] # inverse regularization parameter (smaller C simpler model, larger C more complex model) 
MAX_ITER = 2000 # allows more iterations for convergence, especially with smaller C values that are more regularized and may require more steps to fit well
RANDOM_STATE = 42 # our set seed
CLASS_WEIGHT = "balanced" # auto bias correction for imbalanced classes

# functions to load splits and folds

def load_cv_tfidf_fold(fold_idx):
    fold_tag = f"fold{fold_idx}"
    return {
        "X_train": sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_train_{fold_tag}.npz"),
        "X_val": sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_val_{fold_tag}.npz"),
        "y_train": np.load(ARTIFACT_DIR / f"labels_train_{fold_tag}.npy"),
        "y_val": np.load(ARTIFACT_DIR / f"labels_val_{fold_tag}.npy"),
    }


def load_full_tfidf_split():
    return {
        "X_train_full": sparse.load_npz(ARTIFACT_DIR / "X_tfidf_train_full.npz"),
        "X_test": sparse.load_npz(ARTIFACT_DIR / "X_tfidf_test.npz"),
        "y_train_full": np.load(ARTIFACT_DIR / "labels_train_full.npy"),
        "y_test": np.load(ARTIFACT_DIR / "labels_test.npy"),
    }

# function to compute relevant metrics for evaluation

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


cv_summary_rows = []
cv_fold_details_by_c = {}

for c_value in C_GRID:
    fold_rows = []

    for fold_idx in range(1, N_SPLITS + 1):
        fold_data = load_cv_tfidf_fold(fold_idx)

        model = LogisticRegression(
            C=c_value,
            solver="saga", # algo that finds best coefficient values by minimizing the loss function with regularization
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(fold_data["X_train"], fold_data["y_train"])

        y_val_prob = model.predict_proba(fold_data["X_val"])[:, 1]
        fold_metrics = compute_metrics(fold_data["y_val"], y_val_prob)
        fold_metrics["fold"] = fold_idx
        fold_rows.append(fold_metrics)

    fold_df = pd.DataFrame(fold_rows)
    cv_fold_details_by_c[c_value] = fold_df

    cv_summary_rows.append(
        {
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        }
    )

cv_results = pd.DataFrame(cv_summary_rows).sort_values("mean_auc", ascending=False).reset_index(drop=True)

print("CV summary across C values (ranked by mean AUC):")
display(cv_results)

best_c = float(cv_results.loc[0, "C"])
print(f"\nBest C by CV mean AUC: {best_c}")
print("Fold-by-fold metrics for best C:")
display(cv_fold_details_by_c[best_c])

full_data = load_full_tfidf_split()

final_model = LogisticRegression(
    C=best_c,
    solver="saga",
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
)
final_model.fit(full_data["X_train_full"], full_data["y_train_full"])

y_test_prob = final_model.predict_proba(full_data["X_test"])[:, 1]
test_metrics = compute_metrics(full_data["y_test"], y_test_prob)

test_metrics_df = pd.DataFrame([test_metrics], index=["test"])
print("\nFinal held-out test metrics (TF-IDF + Logistic Regression):")
display(test_metrics_df)

# Keep model and selected hyperparameter available for later notebook cells.
logreg_tfidf_best_c = best_c
logreg_tfidf_model = final_model

CV summary across C values (ranked by mean AUC):


,C,mean_auc,std_auc,mean_f1,mean_precision,mean_recall,mean_accuracy
0,0.10,0.771302,0.002317,0.553948,0.475653,0.663110,0.717944
1,1.00,0.770819,0.002187,0.552971,0.477536,0.656735,0.719557
2,10.00,0.760218,0.002080,0.543451,0.467280,0.649307,0.711858
3,0.01,0.749061,0.002531,0.531866,0.448278,0.653780,0.696026



Best C by CV mean AUC: 0.1
Fold-by-fold metrics for best C:


,auc,f1,precision,recall,accuracy,fold
0,0.772300,0.554841,0.475571,0.665823,0.717800,1
1,0.772204,0.554006,0.476934,0.660789,0.718994,2
2,0.774174,0.557744,0.477161,0.671075,0.718907,3
3,0.769042,0.553200,0.476228,0.659851,0.718476,4
4,0.768789,0.549948,0.472370,0.658014,0.715544,5



Final held-out test metrics (TF-IDF + Logistic Regression):


,auc,f1,precision,recall,accuracy
test,0.768161,0.545063,0.467519,0.653445,0.71838


## 2. Logistic Regression on Lemmatised TF-IDF

In [4]:
def load_cv_tfidf_lemma_fold(fold_idx):
    fold_tag = f"fold{fold_idx}"
    return {
        "X_train": sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_train_{fold_tag}.npz"),
        "X_val": sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_val_{fold_tag}.npz"),
        "y_train": np.load(ARTIFACT_DIR / f"labels_train_{fold_tag}.npy"),
        "y_val": np.load(ARTIFACT_DIR / f"labels_val_{fold_tag}.npy"),
    }


def load_full_tfidf_lemma_split():
    return {
        "X_train_full": sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_train_full.npz"),
        "X_test": sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_test.npz"),
        "y_train_full": np.load(ARTIFACT_DIR / "labels_train_full.npy"),
        "y_test": np.load(ARTIFACT_DIR / "labels_test.npy"),
    }

# function to compute relevant metrics for evaluation

def compute_metrics_lemma(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


cv_summary_rows_lemma = []
cv_fold_details_by_c_lemma = {}

for c_value in C_GRID:
    fold_rows = []

    for fold_idx in range(1, N_SPLITS + 1):
        fold_data = load_cv_tfidf_lemma_fold(fold_idx)

        model = LogisticRegression(
            C=c_value,
            solver="saga", # algo that finds best coefficient values by minimizing the loss function with regularization
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(fold_data["X_train"], fold_data["y_train"])

        y_val_prob = model.predict_proba(fold_data["X_val"])[:, 1]
        fold_metrics = compute_metrics_lemma(fold_data["y_val"], y_val_prob)
        fold_metrics["fold"] = fold_idx
        fold_rows.append(fold_metrics)

    fold_df = pd.DataFrame(fold_rows)
    cv_fold_details_by_c_lemma[c_value] = fold_df

    cv_summary_rows_lemma.append(
        {
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        }
    )

cv_results_lemma = pd.DataFrame(cv_summary_rows_lemma).sort_values("mean_auc", ascending=False).reset_index(drop=True)

print("CV summary across C values (ranked by mean AUC):")
display(cv_results_lemma)

best_c_lemma = float(cv_results_lemma.loc[0, "C"])
print(f"\nBest C by CV mean AUC: {best_c_lemma}")
print("Fold-by-fold metrics for best C:")
display(cv_fold_details_by_c_lemma[best_c_lemma])

full_data_lemma = load_full_tfidf_lemma_split()

final_model_lemma = LogisticRegression(
    C=best_c_lemma,
    solver="saga",
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
)
final_model_lemma.fit(full_data_lemma["X_train_full"], full_data_lemma["y_train_full"])

y_test_prob_lemma = final_model_lemma.predict_proba(full_data_lemma["X_test"])[:, 1]
test_metrics_lemma = compute_metrics_lemma(full_data_lemma["y_test"], y_test_prob_lemma)

test_metrics_df_lemma = pd.DataFrame([test_metrics_lemma], index=["test"])
print("\nFinal held-out test metrics (TF-IDF + lemma + Logistic Regression):")
display(test_metrics_df_lemma)

# Keep model and selected hyperparameter available for later notebook cells.
logreg_tfidf_lemma_best_c = best_c_lemma
logreg_tfidf_lemma_model = final_model_lemma

CV summary across C values (ranked by mean AUC):


,C,mean_auc,std_auc,mean_f1,mean_precision,mean_recall,mean_accuracy
0,0.10,0.770962,0.002212,0.553105,0.474724,0.662490,0.717239
1,1.00,0.770390,0.002003,0.552775,0.476715,0.657723,0.718906
2,10.00,0.760137,0.001919,0.543103,0.466354,0.650099,0.711097
3,0.01,0.748701,0.002523,0.531286,0.447088,0.654556,0.694950



Best C by CV mean AUC: 0.1
Fold-by-fold metrics for best C:


,auc,f1,precision,recall,accuracy,fold
0,0.771914,0.554776,0.475851,0.665089,0.718037,1
1,0.771871,0.553762,0.476149,0.661606,0.718368,2
2,0.773613,0.555278,0.475949,0.666340,0.718088,3
3,0.769273,0.552113,0.474555,0.659973,0.717182,4
4,0.768141,0.549595,0.471118,0.659442,0.714520,5



Final held-out test metrics (TF-IDF + lemma + Logistic Regression):


,auc,f1,precision,recall,accuracy
test,0.767876,0.543753,0.466988,0.65072,0.718069
